# Expense Analyst Agent — Walkthrough

Run **top to bottom**. Need a `.env` with `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`.

1. **Install + imports** — packages, then SDK / PDF / Gradio imports.
2. **Azure setup** — load keys, disable tracing, wire the chat model.
3. **Tools** — PDF text, CSV parse/validate, spending summary by category.
4. **Agent** — `ExpenseAnalystAgent` uses those tools.
5. **Tests** — one PDF run, one CSV run. Check the printed summaries.
6. **Chat** — last code cell opens Gradio. Sample files: `data/MonthlySpendingData_Placeholder.pdf`, `data/sample_transactions.csv`.

## 1. Install packages

Run this once. Installs the Agents SDK, Azure OpenAI client, PDF reader, Gradio, and pandas.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 78 packages in 1.29s
Checked 78 packages in 5ms


## Imports

Brings in dotenv, Azure OpenAI, Agents SDK, PDF reader, Gradio, and pandas. No output expected.

In [2]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

c:\Users\Test\OneDrive\Desktop\Everything\Classes\Fall 2026 Classes\Agentic AI\Final Project\FinalProject_IT3025C-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Azure keys

Reads `.env`. Both prints should be `True`. If not, stop and fix the env file.

In [3]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Wire the Azure model

Turns tracing off, then builds `azure_model` for every agent call below.

In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. PDF tool

Pulls text from `data/MonthlySpendingData_Placeholder.pdf`. `extract_pdf_text` is the tool the agent can call. You should see a character-count print.

In [5]:
def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# AI Agent Tool
@function_tool
def extract_pdf_text(file_path: str) -> str:
    """Extract text content from a PDF file."""
    return read_pdf(file_path)


# PDF Collection
MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": read_pdf(
            "data/MonthlySpendingData_Placeholder.pdf"
        )
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION[
    "monthly_expenses_pdf_1"
]["content"]

print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


## 5. CSV validation

Banks export CSVs differently — this file is a placeholder. Required columns: `date`, `description`, `amount`. Bad rows raise an error.

In [6]:
# Validate csv file
def load_validated_csv(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    required = {"date", "description", "amount"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df[["date", "amount", "description"]].isna().any().any():
        raise ValueError("CSV contains invalid or missing values")

    if df["description"].astype(str).str.strip().eq("").any():
        raise ValueError("CSV contains empty descriptions")

    return df

## 6. CSV parse tool

Wraps the validator and returns transactions as JSON. On a bad file it returns `success: False` plus the error.

In [7]:
@function_tool
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 7. Spending summary tool

Totals spend and buckets merchants into Groceries / Dining / Transportation / Other.

In [8]:
@function_tool
def calculate_spending_summary_tool(file_path: str) -> dict:
    """Calculate total spending and expenses by category from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        # Categorize transactions based on merchant descriptions
        def categorize(description: str) -> str:
            description = description.lower()

            if any(word in description for word in ["walmart", "kroger", "aldi"]):
                return "Groceries"
            elif any(word in description for word in ["starbucks", "mcdonald", "chipotle"]):
                return "Dining"
            elif any(word in description for word in ["shell", "exxon", "uber"]):
                return "Transportation"
            else:
                return "Other"

        df["category"] = df["description"].apply(categorize)

        spending_by_category = (
            df.groupby("category")["amount"]
            .sum()
            .round(2)
            .to_dict()
        )

        return {
            "success": True,
            "total_spending": round(float(df["amount"].sum()), 2),
            "transaction_count": len(df),
            "spending_by_category": spending_by_category
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 8. Save tool (stub)

Placeholder only. Not attached to the agent yet — skip mentally, do not expect it to persist data.

In [12]:
@function_tool
def save_transactions_tool():
    return

## 9. Build ExpenseAnalystAgent

System prompt + three tools: PDF extract, CSV parse, spending summary. Print should say `Agent built: ExpenseAnalystAgent`.

In [41]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user

#Responsibilities
You are to, when prompted by the user, to list the monthly expenses in the files provided, whether that be the pdf of csv files.
then you handoff the data to the BudgetOptimizerAgent. 
Explicitly state what part of the output was provided by the ExpenseAnalystAgent
Explicitly state which tools are used and when.


"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_spending_summary_tool],
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)




Agent built: ExpenseAnalystAgent


## 10. Compare Grocery Prices tool

Tool used by Budget Optimizer Agent. Uses the free Kroger Public API: finds the nearest Kroger store by ZIP, searches a product, and ranks brands by unit price so the agent can suggest a cheaper brand of the same item. Needs `KROGER_CLIENT_ID` and `KROGER_CLIENT_SECRET` in `.env`.

### Kroger setup + token

Reads Kroger keys from `.env` and gets an OAuth2 token (cached until it expires).

In [ ]:
import re
import time
import requests

KROGER_CLIENT_ID = os.getenv("KROGER_CLIENT_ID")
KROGER_CLIENT_SECRET = os.getenv("KROGER_CLIENT_SECRET")
# Production by default; set KROGER_BASE_URL=https://api-ce.kroger.com/v1 in .env for a Certification app
KROGER_BASE_URL = os.getenv("KROGER_BASE_URL", "https://api.kroger.com/v1")

_kroger_token = {"value": None, "expires_at": 0}


def get_kroger_token() -> str:
    """Get (and cache) an OAuth2 client-credentials token for the Kroger API."""
    if _kroger_token["value"] and time.time() < _kroger_token["expires_at"] - 60:
        return _kroger_token["value"]

    response = requests.post(
        f"{KROGER_BASE_URL}/connect/oauth2/token",
        auth=(KROGER_CLIENT_ID, KROGER_CLIENT_SECRET),
        data={"grant_type": "client_credentials", "scope": "product.compact"},
        timeout=10,
    )
    response.raise_for_status()
    body = response.json()
    _kroger_token["value"] = body["access_token"]
    _kroger_token["expires_at"] = time.time() + body["expires_in"]
    return _kroger_token["value"]

### Kroger request helper

Sends an authorized GET request to the Kroger API.

In [ ]:
def kroger_get(path: str, params: dict) -> dict:
    response = requests.get(
        f"{KROGER_BASE_URL}{path}",
        headers={"Authorization": f"Bearer {get_kroger_token()}"},
        params=params,
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

### Unit size parser

Turns sizes like `1 gal`, `1/2 gal`, `2 lb` into one base unit (fl oz, oz, ct) so different brands can be compared fairly.

In [ ]:
# Convert a size string like "64 fl oz", "1 gal", "2 lb", "12 ct" into (amount, base_unit)
UNIT_CONVERSIONS = {
    "oz": (1, "oz"), "lb": (16, "oz"), "g": (0.035274, "oz"), "kg": (35.274, "oz"),
    "fl oz": (1, "fl oz"), "gal": (128, "fl oz"), "qt": (32, "fl oz"), "pt": (16, "fl oz"),
    "l": (33.814, "fl oz"), "ml": (0.033814, "fl oz"),
    "ct": (1, "ct"),
}


def parse_size(size: str):
    match = re.match(r"\s*(\d+(?:\.\d+)?)(?:/(\d+))?\s*(fl oz|oz|lb|gal|qt|pt|ml|kg|g|l|ct)\b", size.lower())
    if not match:
        return None
    amount = float(match.group(1)) / (float(match.group(2)) if match.group(2) else 1)
    factor, base_unit = UNIT_CONVERSIONS[match.group(3)]
    return amount * factor, base_unit

### Product summary

Keeps only priced products and builds rows: brand, size, price (sale price if lower), and unit price.

In [ ]:
def summarize_products(products: list) -> list:
    """Flatten Kroger product results into brand / price / unit-price rows."""
    rows = []
    for product in products:
        item = (product.get("items") or [{}])[0]
        price_info = item.get("price") or {}
        regular = price_info.get("regular")
        if not regular:
            continue  # no price at this store
        promo = price_info.get("promo")
        price = promo if promo and promo < regular else regular
        size = item.get("size", "")
        parsed = parse_size(size)

        rows.append({
            "brand": product.get("brand", "Unknown"),
            "category": (product.get("categories") or ["Unknown"])[0],
            "description": product.get("description", ""),
            "size": size,
            "price": round(price, 2),
            "on_sale": price < regular,
            "unit_price": round(price / parsed[0], 3) if parsed and parsed[0] else None,
            "unit": parsed[1] if parsed else None,
        })
    return rows

### Saved price data (cache)

Every live Kroger lookup is saved to `data/kroger_price_cache.json`. Without a Kroger API key, the AI can still use the tool - it reads from this file instead.

In [ ]:
import json
from datetime import date

KROGER_CACHE_PATH = "data/kroger_price_cache.json"


def load_price_cache() -> dict:
    if not os.path.exists(KROGER_CACHE_PATH):
        return {}
    with open(KROGER_CACHE_PATH) as f:
        return json.load(f)


def save_to_price_cache(product_name: str, entry: dict) -> None:
    cache = load_price_cache()
    cache[product_name.lower().strip()] = entry
    with open(KROGER_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)

### Live Kroger lookup

Nearest Kroger store by ZIP → search the product → summarized rows. Saves the result to the cache.

In [ ]:
def fetch_kroger_prices(product_name: str, zip_code: str) -> dict:
    locations = kroger_get("/locations", {"filter.zipCode.near": zip_code, "filter.limit": 1})["data"]
    if not locations:
        raise ValueError(f"No Kroger store found near {zip_code}")
    store = locations[0]

    products = kroger_get("/products", {
        "filter.term": product_name,
        "filter.locationId": store["locationId"],
        "filter.limit": 20,
    })["data"]

    entry = {
        "store": f"{store['name']} ({store['address']['addressLine1']}, {store['address']['city']})",
        "zip_code": zip_code,
        "fetched_on": date.today().isoformat(),
        "rows": summarize_products(products),
    }
    save_to_price_cache(product_name, entry)
    return entry

### Rank brands

Keeps only products in the most common category (search also returns unrelated items), then compares by unit price (using the most common unit) so different sizes are fair; falls back to shelf price.

In [ ]:
def rank_brands(rows: list) -> dict:
    # Keyword search also returns unrelated items (e.g. "peanut butter" -> Reese's candy),
    # so only compare products in the most common category
    categories = [r.get("category", "Unknown") for r in rows]
    main_category = max(set(categories), key=categories.count)
    rows = [r for r in rows if r.get("category", "Unknown") == main_category]

    units = [r["unit"] for r in rows if r["unit"]]
    main_unit = max(set(units), key=units.count) if units else None
    comparable = [r for r in rows if main_unit and r["unit"] == main_unit]

    if comparable:
        ranked, basis, key = sorted(comparable, key=lambda r: r["unit_price"]), f"price per {main_unit}", "unit_price"
    else:
        ranked, basis, key = sorted(rows, key=lambda r: r["price"]), "shelf price (sizes not comparable)", "price"

    cheapest, priciest = ranked[0], ranked[-1]
    savings_pct = round((1 - cheapest[key] / priciest[key]) * 100, 1) if priciest[key] else 0

    return {
        "category": main_category,
        "compared_by": basis,
        "cheapest_brand": cheapest,
        "most_expensive_brand": priciest,
        "savings_percentage": savings_pct,
        "all_options": ranked[:10],
    }

### `compare_grocery_prices_tool`

The tool the agent calls. With a Kroger key → live prices (and saves them). Without a key → saved prices from `data/kroger_price_cache.json`.

In [ ]:
@function_tool
def compare_grocery_prices_tool(product_name: str, zip_code: str = "45221") -> dict:
    """Compare prices of the same grocery product across different brands at a Kroger store.
    Returns brands sorted from cheapest to most expensive per unit (e.g., per oz), so a cheaper brand can be suggested."""
    has_key = bool(KROGER_CLIENT_ID and KROGER_CLIENT_SECRET)

    if has_key:
        try:
            entry = fetch_kroger_prices(product_name, zip_code)
        except (requests.RequestException, KeyError, ValueError) as e:
            return {"success": False, "error": str(e)}
        source = "live Kroger API"
    else:
        entry = load_price_cache().get(product_name.lower().strip())
        if not entry:
            return {
                "success": False,
                "error": f"No Kroger API key and no saved prices for '{product_name}'.",
                "saved_products": sorted(load_price_cache()),
            }
        source = f"saved Kroger data from {entry['fetched_on']}"

    if not entry["rows"]:
        return {"success": False, "error": f"No priced results for '{product_name}' at {entry['store']}"}

    return {
        "success": True,
        "source": source,
        "store": entry["store"],
        "product": product_name,
        **rank_brands(entry["rows"]),
    }

### Build saved price data (only with a Kroger key)

Run once with a Kroger key to save prices for common groceries, then commit `data/kroger_price_cache.json`. Without a key this cell just skips.

In [ ]:
COMMON_GROCERIES = [
    "milk", "eggs", "bread", "butter", "cheese", "yogurt", "white rice", "pasta", "cereal",
    "chicken breast", "ground beef", "ground coffee", "peanut butter", "orange juice",
]

if KROGER_CLIENT_ID and KROGER_CLIENT_SECRET:
    for item in COMMON_GROCERIES:
        try:
            # 45221 is Cincinnati zipcode
            entry = fetch_kroger_prices(item, "45221")
            print(f"Saved {item}: {len(entry['rows'])} priced products")
        except (requests.RequestException, KeyError, ValueError) as e:
            print(f"Failed {item}: {e}")
else:
    print("No Kroger key — skipping. Saved products:", sorted(load_price_cache()))

## 11. Identify Recurring Expenses Tool

Tool used by Budget Optimizer

In [43]:
## identify_recurring_expenses_Tool
@function_tool
def identify_recurring_expenses_Tool(Expenses_category: str)-> dict:
     """Locate Recurring expenses in database to see what the person recurrently spends on(e.g., Food, Water, tools)."""
     Expenses_tiers = {
        "Food":{
            "Quantity_Size" "Standard Supermarket (Kroger/Meijer)"
        },
        "Water":{
            "Quality_rating" "Local Gas stations (Speedway/Marathon)"
        },
        "Tools": {
            "Customer_Rating" "Hardware stores(Harbor Freight/Home)"
        }
    }

     if Expenses_category in Expenses_tiers:
        return  {"success": True, "results": Expenses_tiers[Expenses_category]}  
     return {"success": False, "message": "No expenses are recurring in this category"}

## 12. Calculate Budget Adjustments tool

Tool used by Budget Optimizer. Estimates how much the user could save by reducing spending in one category: new monthly budget plus monthly and annual savings.

In [ ]:
## calculate_budget_adjustments_tool
@function_tool
def calculate_budget_adjustments_tool(category: str, current_monthly_amount: float, reduction_rate: float) -> dict:
    """Calculate the new monthly budget and monthly/annual savings for one category.
    reduction_rate is a fraction between 0 and 1 (e.g. 0.20 = cut 20%)."""
    if current_monthly_amount < 0:
        return {"success": False, "error": "current_monthly_amount must be >= 0"}
    if not 0 <= reduction_rate <= 1:
        return {"success": False, "error": "reduction_rate must be between 0 and 1"}

    monthly_savings = round(current_monthly_amount * reduction_rate, 2)

    return {
        "success": True,
        "category": category,
        "current_monthly_amount": round(current_monthly_amount, 2),
        "reduction_rate": reduction_rate,
        "new_monthly_budget": round(current_monthly_amount - monthly_savings, 2),
        "monthly_savings": monthly_savings,
        "annual_savings": round(monthly_savings * 12, 2),
    }

## Agent 2: Build Budget Optimizer
System prompt + three tools: PDF extract, CSV parse, Budget Plan. Print should say `Agent built: Budget Optimizer`.

In [ ]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are the Budget Optimizer Agent, a specialized financial advisor within a multi-agent personal finance service.
Your primary role is to take the structured expense and spending summary provided by the Expense Analyst Agent and identify unnecessary spending, recurring subscription leaks, and high-cost categories.

## responibilities
Analyze Category Breakdown: Examine spending across categories like Groceries, Dining, Transportation, Utilities, Other.
Detect Recurring Leaks: Use your tools to flag subscriptions, recurring bills, and minor purchases.
Compare Alternatives: The spending summary only shows grocery totals, not individual items. If Groceries spending appears, call compare_grocery_prices_tool for 2-3 common staples (e.g., milk, eggs, bread) to show the user a cheaper brand of the same item (compared by unit price). If the user names a specific product, compare that product instead. If the tool fails and returns saved_products, retry with items from that list. Never call the tool with a category name like "Groceries".
Calculate Adjustments: Use calculate_budget_adjustments_tool to compute the new budget and potential monthly and annual savings for each category you suggest reducing. Provide concrete, actionable budget adjustments.

##Rules
Do not make up numbers, base your calculations on the data provided by the Expense Analyst or retrieved via your tools.
Be constructive and realistic: Avoid shaming the user. Provide practical steps like reducing dining out by 20% rather than cutting it to zero.
Format clearly, Use bullet points, bold text for dollar amounts, and clear step-by-step optimization layouts.
Explicitly say which part of the output was provided by the BudgetOptimizerAgent.
Explicitly state which tools are used by the BudgetOptimizerAgent and when.
"""
BudgetOptimizerAgent = Agent(
    name="BudgetOptimizerAgent",
    handoff_description="Create and print a layout plan for spending.",
    instructions=instructions,
    tools=[compare_grocery_prices_tool, identify_recurring_expenses_Tool, calculate_budget_adjustments_tool],
    model=azure_model,
)

print("Agent built:", BudgetOptimizerAgent.name)


Agent built: BudgetOptimizerAgent


## Prompt Chaining 
Working for the Expense Analyst to the Budget Optimizer.

In [ ]:
from agents import Agent, Runner, SQLiteSession, function_tool
import pandas as pd

chat_session = SQLiteSession("gradio_user", "user_expenses.db")
async def chained_gradio_chat(message, history):
   
    analyst_result = await Runner.run(
        ExpenseAnalystAgent, 
        message, 
        session=chat_session
    )
    analyst_summary = analyst_result.final_output
    optimizer_prompt = (
        f"Her is the verified spending summary and expense analysis from the Expense Analyst Agent:\n\n"
        f"{analyst_summary}\n\n"
        f"Please analyze this data, identify recurring expenses or unnecessary spending leaks, "
        f"and provide a clear, step by step budget optimization plan for the user."
    )
    
    optimizer_result = await Runner.run(
        BudgetOptimizerAgent, 
        optimizer_prompt, 
        session=chat_session
    )
    
    return optimizer_result.final_output

print("Chained function and session ready.")

Chained function and session ready.


## 10. Test — PDF

Asks the agent to read the placeholder PDF and summarize expenses. Expect a list of bills plus optimization tips.

In [15]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/MonthlySpendingData_Placeholder.pdf and summarize my expenses."
)

print(result.final_output)

Here’s a summary of your monthly expenses from the PDF:

### Monthly expenses
- Rent: $1,000
- Gas: $200
- Groceries: $400
- Spotify: $10
- Netflix: $15
- Take-out: $50
- Car Payment: $160
- Car Insurance: $100
- Medical Insurance: $300
- Amazon orders: $100
- Miscellaneous: $400

### Total monthly spending
- **$3,500**

### Additional income note
- Money taken home: **$765**

## Quick spending observations
Your biggest expense categories are:
1. **Rent — $1,000**
2. **Groceries — $400**
3. **Miscellaneous — $400**
4. **Medical Insurance — $300**
5. **Gas — $200**

## Ways to optimize your spending
- **Review miscellaneous spending ($400):** This is one of the largest categories and may include reducible expenses.
- **Cut back on Amazon orders ($100):** Setting a monthly cap could help.
- **Reduce groceries ($400):** Meal planning, buying store brands, and shopping with a list may lower costs.
- **Limit take-out ($50):** Already modest, but reducing it further can help.
- **Evaluate su

## 11. Test — CSV

Asks the agent to read `data/sample_transactions.csv`. Expect a transaction count, total, and category breakdown.

In [16]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/sample_transactions.csv")

print(result.final_output)

I read `data/sample_transactions.csv`.

Summary:
- Total spending: $1,642.54
- Number of transactions: 25

By category:
- Other: $1,267.43
- Groceries: $210.86
- Transportation: $112.75
- Dining: $51.50

Quick insight:
- Most of your spending is in Other, which is by far the largest category.
- Groceries are your next biggest expense.
- Dining is relatively low.

Optimization ideas:
- Review the Other category first, since that’s where most of your money is going. It may contain recurring bills, shopping, or uncategorized purchases that could be reduced.
- If you want tighter control, recategorizing the Other transactions would help identify specific savings opportunities.
- Groceries are moderate, so using meal planning or store budgeting could help reduce them further.
- Transportation looks manageable, but you could still watch for ride-share or fuel costs if applicable.

If you want, I can also:
- parse and list the individual transactions from the CSV, or
- give you a more detaile

## 12. Gradio chat

Opens a chat UI in the browser. Session memory goes to `user_expenses.db`. Try: *Read data/sample_transactions.csv* or *summarize my PDF expenses*.

In [52]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")
gr.ChatInterface(chained_gradio_chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Notes (from Chris)

Skeleton agent: one PDF for now, tools first, memory later. Last cell is teammate comments only.

In [18]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris